# Darmanis et al. (2017) -- Glioblastoma scRNA-seq

**Source:** Darmanis et al., *Cell Reports* 21.5 (2017): 1399--1410.

3,589 single cells from human glioblastoma tumors, FACS-sorted into 6 cell populations. Features: 500 highly variable genes. Moderate imbalance (9:1 ratio).

---

## 0. Imports & Configuration

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "notebooks" / "benchmarking_code"))

In [ ]:
import multiprocessing
import os, random

from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from carve import CARVE

from case_study_plotting import (
    baseline_metrics_over_k,
    build_baseline_best_labels,
    extract_ari_comparison,
    plot_ari_comparison_lollipop,
    plot_ari_comparison_bar,
    plot_ari_comparison_dotplot,
    plot_composite_figure,
    plot_dim_red,
    alluvial_compare,
)

In [ ]:
# --- Configuration ---
SUBSAMPLE = False
RANDOM_SEED = 42
K = 15

MEASURE = "s"
RULE = "1se"
NOT_TWO = True

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

def get_n_jobs():
    env_val = os.environ.get("CARVE_N_JOBS", None)
    if env_val is not None:
        try:
            return int(env_val)
        except ValueError:
            pass
    try:
        n_cores = multiprocessing.cpu_count()
    except NotImplementedError:
        n_cores = 4
    return max(1, n_cores - 1)

N_JOBS = get_n_jobs()
print(f"Using N_JOBS={N_JOBS} for parallel processing.")

In [ ]:
%env PYTHONWARNINGS=ignore::FutureWarning,ignore::DeprecationWarning,ignore::UserWarning,ignore::RuntimeWarning

---

## 1. Dataset Overview

In [ ]:
data_dir = Path("../../data/DARMANIS Data")
df = pd.read_csv(data_dir / "GBM_HVG500_with_metadata.csv", index_col=0)

# Separate gene expression from metadata
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
qc_keywords = ["reads", "Reads", "Splice", "Length", "Percent", "percent", "length", "sites"]
gene_cols = [c for c in numeric_cols if not any(kw in c for kw in qc_keywords)]

X_raw = df[gene_cols].values
y_raw = df["Selection"]

print(f"Cells: {X_raw.shape[0]:,}")
print(f"Features: {X_raw.shape[1]:,}")
print(f"Cell types: {y_raw.nunique()}")
print()
print("Class distribution:")
print(y_raw.value_counts().to_string())

---

## 2. Preprocessing

In [ ]:
X = PCA(n_components=min(50, X_raw.shape[1]), random_state=RANDOM_SEED).fit_transform(
    StandardScaler().fit_transform(X_raw)
)
y = y_raw
print(f"After PCA: {X.shape}")

In [ ]:
plot_dim_red(X=X, y=y, method="pca", title="PCA of Darmanis GBM (colored by cell type)")

In [ ]:
plot_dim_red(X=X, y=y, method="tsne", title="t-SNE of Darmanis GBM (colored by cell type)")

---

## 3. Baseline Metrics

In [ ]:
ks = list(range(2, K + 1))

model_grids = [
    (KMeans, {"n_clusters": ks}),
    (AgglomerativeClustering, {"n_clusters": ks, "linkage": ["ward", "single"]}),
]

In [ ]:
curves_df, best_df = baseline_metrics_over_k(
    X, y=y, model_grids=model_grids,
    metrics=["silhouette", "gap", "DB", "CH"],
    n_jobs=N_JOBS, random_state=RANDOM_SEED, ncols=2, figsize=(14, 9),
)

In [ ]:
print("Best k selected by each metric:")
print(best_df[["metric", "best_model", "best_k", "best_ari"]].to_string(index=False))

---

## 4. CARVE Analysis

In [ ]:
carve_path = Path("./carve_state_saves/carve_darmanis.carve")

if not carve_path.is_file():
    carve = CARVE(
        estimator_param_grids=model_grids, n_jobs=N_JOBS, random_state=RANDOM_SEED
    )
    carve.fit(X, reference_labels=y)
    carve_path.parent.mkdir(parents=True, exist_ok=True)
    carve.save(str(carve_path))
    print(f"Saved CARVE state to {carve_path}")
else:
    print(f"Loading CARVE state from {carve_path}")
    carve = CARVE.load(str(carve_path))
    carve.X_ = X

In [ ]:
carve.plot_metric_over_n_clusters(measure=MEASURE, rule=RULE, not_two=NOT_TWO)

In [ ]:
carve.plot_metric_over_n_clusters(measure="g", rule=RULE, not_two=NOT_TWO)

In [ ]:
carve.plot_consensus_matrix(measure=MEASURE, rule=RULE, not_two=NOT_TWO)

In [ ]:
carve.plot_cluster_violin(measure=MEASURE, rule=RULE, not_two=NOT_TWO)

In [ ]:
print(f"CARVE selects k={carve.get_k(measure=MEASURE, rule=RULE, not_two=NOT_TWO)}")

---

## 5. Quantitative Comparison

### 5.1 Composite Figure

In [ ]:
silhouette_labels, sil_model, sil_k = build_baseline_best_labels(
    X, best_df, model_grids, metric="silhouette", random_state=RANDOM_SEED
)
print(f"Silhouette selects: {sil_model}, k={sil_k}")

In [ ]:
fig = plot_composite_figure(
    X, y, carve, curves_df, best_df, silhouette_labels,
    measure=MEASURE, rule=RULE,
    carve_measures=[("stability", "1se"), ("generalizability", "1se")],
    normalize_baseline=True,
    true_label_legend_title="Cell Type",
    carve_title="CARVE clustering", baseline_title="Silhouette clustering",
    alluvial_left_title="CARVE", alluvial_right_title="Silhouette",
    alluvial_true_title="True Labels", show_alluvial=True,
    figsize=(18, 16), scatter_s=10,
)

### 5.2 ARI with Ground Truth

In [ ]:
ari_df = extract_ari_comparison(
    y_true=y, best_df=best_df, carve_obj=carve, X=X,
    carve_measures=[("stability", "1se"), ("generalizability", "1se")],
    not_two=NOT_TWO,
)
print(ari_df.to_string(index=False))

In [ ]:
plot_ari_comparison_lollipop(ari_df)

In [ ]:
plot_ari_comparison_bar(ari_df)

In [ ]:
plot_ari_comparison_dotplot(ari_df)

### 5.3 Alluvial Diagram

In [ ]:
carve_labels = carve.get_labels(measure=MEASURE, rule=RULE, not_two=NOT_TWO)
fig = alluvial_compare(
    y_true=y, left_labels=carve_labels, right_labels=silhouette_labels,
    left_title="CARVE", right_title="Silhouette", true_title="True Labels",
    palette_name="tab10", link_alpha=0.35,
)
fig.show()

---

## 6. Summary

Darmanis GBM: 3,589 cells, 500 HVGs, 6 FACS-sorted populations from glioblastoma with moderate imbalance (9:1).

---